In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
from nemosis import static_table
from datetime import timedelta
import dask.dataframe as dd
from concurrent.futures import ProcessPoolExecutor
import time

raw_data_cache = '/Volumes/T7/NEMO-misc'

Found 1448 Feather files in /Volumes/T7/bid-volume-data-feather-sorted.
First file: PUBLIC_DVD_BIDPEROFFER_D_20090701.feather
Last file:  PUBLIC_DVD_BIDPEROFFER_D_20241112.feather
[1/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20090701.feather ✓  (elapsed: 27.7s)
[2/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20090801.feather ✓  (elapsed: 56.0s)
[3/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20090901.feather ✓  (elapsed: 82.8s)
[4/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20091001.feather ✓  (elapsed: 111.4s)
[5/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20091101.feather ✓  (elapsed: 139.6s)
[6/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20091201.feather ✓  (elapsed: 168.4s)
[7/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20100101.feather ✓  (elapsed: 196.2s)
[8/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20100201.feather ✓  (elapsed: 220.5s)
[9/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20100301.feather ✓  (elapsed: 254.6s)
[10/1448] Converting: PUBLIC_DVD_BIDPEROFFER_D_20100401.feather ✓  

In [9]:
# Join DUIDs with firm names and further info
generator_info_df = static_table(table_name='Generators and Scheduled Loads', 
                              raw_data_location=raw_data_cache,
                              update_static_file=False)
generator_info_df

INFO: Retrieving static table Generators and Scheduled Loads


,Participant,Station Name,Region,Dispatch Type,Category,Classification,Fuel Source - Primary,Fuel Source - Descriptor,Technology Type - Primary,Technology Type - Descriptor,Aggregation,DUID
0,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,ADPBA1G
1,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Load,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,ADPBA1L
2,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Hydro,Water,Renewable,Run of River,Y,ADPMH1
3,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Semi-Scheduled,Solar,Solar,Renewable,Photovoltaic Tracking Flat panel,Y,ADPPV1
4,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Solar,Solar,Renewable,Photovoltaic Flat panel,Y,ADPPV2
...,...,...,...,...,...,...,...,...,...,...,...,...
527,Tailem Bend II Project Company Pty Ltd as trus...,Tailem Bend 2 Hybrid Renewable Power Station,SA1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,TB2B1
528,AGL Macquarie Pty Limited,Broken Hill Battery Energy Storage System,NSW1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,BHB1
529,AGL SA Generation Pty Limited,Torrens Island BESS,SA1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,TIB1
530,Capital Battery Pty Ltd as Trustee for Capital...,Capital Battery,NSW1,Load,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,CAPBES1


In [ ]:
parquet_dir = "/Volumes/T7/bid-volume-data-parquet-sorted"

all_files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
# Exclude any files starting with '._'
valid_files = [f for f in all_files if not os.path.basename(f).startswith("._")]

ddf = dd.read_parquet(valid_files)
print(ddf.head())

        SETTLEMENTDATE    DUID    BIDTYPE            OFFERDATE MAXAVAIL  \
0  2009/07/01 00:00:00  AGLHAL     ENERGY  2009/06/19 14:55:14      183   
1  2009/07/01 00:00:00  AGLSOM     ENERGY  2009/06/30 11:23:49      150   
2  2009/07/01 00:00:00  ANGAS1     ENERGY  2009/06/30 11:23:51       30   
3  2009/07/01 00:00:00  ANGAS2     ENERGY  2009/06/30 11:23:51       20   
4  2009/07/01 00:00:00   APD01  LOWER5MIN  2002/10/17 08:51:56        0   

  ENABLEMENTMIN ENABLEMENTMAX LOWBREAKPOINT HIGHBREAKPOINT BANDAVAIL1  \
0             0             0             0              0          0   
1             0             0             0              0          0   
2             0             0             0              0          0   
3             0             0             0              0          0   
4             0             0             0              0          0   

  BANDAVAIL2 BANDAVAIL3 BANDAVAIL4 BANDAVAIL5 BANDAVAIL6 BANDAVAIL7  \
0          0          0          0     

In [10]:
# Directories
input_dir = "/Volumes/T7/bid-volume-data-parquet-sorted"

# Output directories
merged_output_dir = "/Volumes/T7/enriched_merged"
os.makedirs(merged_output_dir, exist_ok=True)

# 1) Load generator_info_df into a small pandas or Dask DataFrame
#    If generator_info_df is already in memory as a pandas DF, skip this read.
import pandas as pd
generator_info_ddf = dd.from_pandas(generator_info_df, npartitions=1)

# 2) Collect all Parquet files, skipping hidden ones
parquet_files = sorted(
    f for f in glob.glob(os.path.join(input_dir, "*.parquet"))
    if not os.path.basename(f).startswith("._")
)

print(f"Found {len(parquet_files)} parquet files in {input_dir}.")

# 3) Read them all into a single Dask DataFrame
ddf = dd.read_parquet(parquet_files)
print("Dask DataFrame loaded. Now merging with generator_info_df...")

# 4) Merge on 'DUID'
enriched_ddf = ddf.merge(generator_info_ddf, on='DUID', how='left')

# 5) Write out the merged dataset (still a Dask DF)
print(f"Writing merged data to {merged_output_dir} ...")
enriched_ddf.to_parquet(merged_output_dir, overwrite=True)
print("Merge complete!")

Found 1448 parquet files in /Volumes/T7/bid-volume-data-parquet-sorted.
Dask DataFrame loaded. Now merging with generator_info_df...
Writing merged data to /Volumes/T7/enriched_merged ...
Merge complete!


In [13]:
merged_output_dir = "/Volumes/T7/enriched_merged"
energy_excluded_dir = "/Volumes/T7/enriched_excluding_energy"
os.makedirs(energy_excluded_dir, exist_ok=True)

import glob, os
parquet_files = [
    f for f in glob.glob(os.path.join(merged_output_dir, "*.parquet"))
    if not os.path.basename(f).startswith("._")
]
enriched_ddf = dd.read_parquet(parquet_files)

print("Loaded merged dataset from disk.")

# 2) Filter out BIDTYPE=ENERGY
filtered_ddf = enriched_ddf[enriched_ddf["BIDTYPE"] != "ENERGY"]
print("Filtering out BIDTYPE == 'ENERGY'...")

# 3) Write to a new folder
print(f"Writing non-ENERGY data to {energy_excluded_dir} ...")
filtered_ddf.to_parquet(energy_excluded_dir, overwrite=True)
print("Non-ENERGY filter complete!")

Loaded merged dataset from disk.
Filtering out BIDTYPE == 'ENERGY'...
Writing non-ENERGY data to /Volumes/T7/enriched_excluding_energy ...
Non-ENERGY filter complete!


In [ ]:
import dask.dataframe as dd
import os

# Define paths
sample_file = '/Volumes/T7/enriched_excluding_energy/part.2.parquet'
output_dir = '/Volumes/T7/enriched_sa1_sample/filtered_sample.parquet/part.0.parquet'
output_file = os.path.join(output_dir, 'filtered_sample.parquet')

# Step 1: Read the sample file
ddf = dd.read_parquet(sample_file)
print("Sample file loaded.")

# Step 2: Inspect the data
print("Data types:\n", ddf.dtypes)
print("First few rows:\n", ddf.head())

# Step 3: Apply filtering
filtered_ddf = ddf[(ddf['BIDTYPE'] != 'ENERGY') & (ddf['Region'] == 'SA1')]
print("Filtering applied.")

# Step 4: Verify the filtered data
print("Filtered data types:\n", filtered_ddf.dtypes)
print("First few rows of filtered data:\n", filtered_ddf.head())

# Step 5: Write the filtered data
filtered_ddf.to_parquet(output_file, overwrite=True)
print(f"Filtered data written to {output_file}")

# Step 6: Review the output
review_ddf = dd.read_parquet(output_file)
print("Filtered file reloaded.")
print("First few rows of the filtered file:\n", review_ddf.head())

Sample file loaded.
Data types:
 SETTLEMENTDATE                  object
DUID                            object
BIDTYPE                         object
OFFERDATE                       object
MAXAVAIL                        object
ENABLEMENTMIN                   object
ENABLEMENTMAX                   object
LOWBREAKPOINT                   object
HIGHBREAKPOINT                  object
BANDAVAIL1                      object
BANDAVAIL2                      object
BANDAVAIL3                      object
BANDAVAIL4                      object
BANDAVAIL5                      object
BANDAVAIL6                      object
BANDAVAIL7                      object
BANDAVAIL8                      object
BANDAVAIL9                      object
BANDAVAIL10                     object
INTERVAL_DATETIME               object
Participant                     object
Station Name                    object
Region                          object
Dispatch Type                   object
Category                       

NotADirectoryError: [Errno 20] Not a directory: '/Volumes/T7/enriched_sa1_sample/filtered_sample.parquet/part.0.parquet/filtered_sample.parquet'

In [ ]:
output_dir = '/Volumes/T7/enriched_sa1_sample/filtered_sample.parquet/part.0.parquet'

ddf = dd.read_parquet(output_dir)
print("Sample file loaded.")

# Step 2: Inspect the data
print("Data types:\n", ddf.dtypes)
print("First few rows:\n", ddf.head())

Sample file loaded.
Data types:
 SETTLEMENTDATE                  object
DUID                            object
BIDTYPE                         object
OFFERDATE                       object
MAXAVAIL                        object
ENABLEMENTMIN                   object
ENABLEMENTMAX                   object
LOWBREAKPOINT                   object
HIGHBREAKPOINT                  object
BANDAVAIL1                      object
BANDAVAIL2                      object
BANDAVAIL3                      object
BANDAVAIL4                      object
BANDAVAIL5                      object
BANDAVAIL6                      object
BANDAVAIL7                      object
BANDAVAIL8                      object
BANDAVAIL9                      object
BANDAVAIL10                     object
INTERVAL_DATETIME               object
Participant                     object
Station Name                    object
Region                          object
Dispatch Type                   object
Category                       

In [ ]:
import glob
import os
import time
import dask.dataframe as dd
import pandas as pd
from datetime import datetime

def load_valid_parquet_files(folder_pattern):
    """
    Load all valid parquet files matching the given pattern into a Dask DataFrame.
    Excludes hidden files (like macOS ._* files).
    
    Args:
        folder_pattern: Glob pattern to match parquet files
        
    Returns:
        Dask DataFrame containing the data from all valid parquet files
    """
    # Get all files matching the pattern
    all_files = glob.glob(folder_pattern)
    
    # Filter out hidden files (like macOS ._ files)
    valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]
    
    # Sort files to ensure consistent reading order
    valid_files = sorted(valid_files)
    
    print(f"Found {len(valid_files)} valid parquet files from pattern: {folder_pattern}")
    
    if not valid_files:
        raise ValueError(f"No valid parquet files found for pattern: {folder_pattern}")
    
    # Load files into a Dask DataFrame
    return dd.read_parquet(valid_files, engine='pyarrow')

def filter_and_merge_data(volume_ddf, price_ddf, output_dir):
    """
    Filter data for 2017-2018 and merge volume and price data.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("Converting date columns to datetime format...")
    start_time = time.time()
    
    # Convert SETTLEMENTDATE to datetime
    volume_ddf = volume_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(volume_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    price_ddf = price_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(price_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    
    print(f"Date conversion defined in {time.time() - start_time:.2f} seconds")
    
    # Filter for 2017-2018 data
    print("Filtering for 2017-2018 data...")
    start_time = time.time()
    
    start_date = pd.Timestamp('2017-01-01')
    end_date = pd.Timestamp('2018-12-31 23:59:59')
    
    volume_filtered = volume_ddf[
        (volume_ddf['SETTLEMENTDATE'] >= start_date) & 
        (volume_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    price_filtered = price_ddf[
        (price_ddf['SETTLEMENTDATE'] >= start_date) & 
        (price_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    print(f"Filtering defined in {time.time() - start_time:.2f} seconds")
    
    # Persist the filtered DataFrames to avoid recomputation
    print("Persisting filtered data (this may take some time)...")
    start_time = time.time()
    
    volume_filtered = volume_filtered.persist()
    price_filtered = price_filtered.persist()
    
    # Wait for persist to complete
    vol_npartitions = volume_filtered.npartitions
    price_npartitions = price_filtered.npartitions
    
    print(f"Filtered volume data: {vol_npartitions} partitions")
    print(f"Filtered price data: {price_npartitions} partitions")
    print(f"Persistence completed in {time.time() - start_time:.2f} seconds")
    
    # Process one partition at a time
    for i in range(vol_npartitions):
        print(f"Processing volume partition {i+1}/{vol_npartitions}...")
        start_time = time.time()
        
        try:
            # Get one volume partition as pandas DataFrame
            vol_part = volume_filtered.get_partition(i).compute()
            print(f"Loaded volume partition with {len(vol_part)} rows in {time.time() - start_time:.2f} seconds")
            
            # Process each price partition
            for j in range(price_npartitions):
                print(f"  Processing price partition {j+1}/{price_npartitions}...")
                part_start = time.time()
                
                try:
                    # Get one price partition as pandas DataFrame
                    price_part = price_filtered.get_partition(j).compute()
                    print(f"  Loaded price partition with {len(price_part)} rows in {time.time() - part_start:.2f} seconds")
                    
                    # Merge the DataFrames
                    merge_start = time.time()
                    merged_df = vol_part.merge(
                        price_part,
                        on=["SETTLEMENTDATE", "DUID", "BIDTYPE", "BIDBAND"],
                        how="inner",
                        suffixes=('_volume', '_price')
                    )
                    print(f"  Merged to {len(merged_df)} rows in {time.time() - merge_start:.2f} seconds")
                    
                    # If we have merged data, write it to a parquet file
                    if not merged_df.empty:
                        save_start = time.time()
                        output_file = os.path.join(output_dir, f"merged_2017_2018_v{i}_p{j}.parquet")
                        merged_df.to_parquet(output_file, engine='pyarrow', index=False)
                        print(f"  Wrote {len(merged_df)} rows to {output_file} in {time.time() - save_start:.2f} seconds")
                    else:
                        print(f"  No matching data between these partitions")
                    
                except Exception as e:
                    print(f"  Error processing price partition {j}: {str(e)}")
                    continue
                
        except Exception as e:
            print(f"Error processing volume partition {i}: {str(e)}")
            continue

def main():
    # Set up paths for input files
    volume_pattern = "/Volumes/T7/bid-volume-melted-files-A4/*.parquet"
    price_pattern = "/Volumes/T7/bid-price-melted-files-B3/*.parquet"
    
    print("Loading volume data metadata...")
    volume_ddf = load_valid_parquet_files(volume_pattern)
    print(f"Volume data columns: {list(volume_ddf.columns)}")
    print(f"Volume data partitions: {volume_ddf.npartitions}")
    
    print("Loading price data metadata...")
    price_ddf = load_valid_parquet_files(price_pattern)
    print(f"Price data columns: {list(price_ddf.columns)}")
    print(f"Price data partitions: {price_ddf.npartitions}")
    
    # Create output directory if it doesn't exist
    output_dir = "/Volumes/T7/bid-merged-fcas-2017-2018"
    print(f"Will save merged data to {output_dir}...")
    
    # Filter and merge data
    filter_and_merge_data(volume_ddf, price_ddf, output_dir)
    
    # Report completion
    print("All done!")
    print(f"Output saved to: {output_dir}")

if __name__ == "__main__":
    main()

In [5]:
import glob
import os
import dask.dataframe as dd

merged_output_dir = "/Volumes/T7/enriched_excluding_energy"
sa1_output_dir = "/Volumes/T7/enriched_sa1"
os.makedirs(sa1_output_dir, exist_ok=True)

# Collect all Parquet files, excluding hidden ones
parquet_files = [
    f for f in glob.glob(os.path.join(merged_output_dir, "*.parquet"))
    if not os.path.basename(f).startswith("._")
]

# Ensure there are valid Parquet files to process
if not parquet_files:
    print("No valid Parquet files found in the specified directory.")
else:
    # Read the Parquet files into a Dask DataFrame
    enriched_ddf = dd.read_parquet(parquet_files)
    print("Loaded merged dataset from disk.")

    # Filter for Region=SA1
    sa_ddf = enriched_ddf[enriched_ddf["Region"] == "SA1"]
    print("Filtering for Region == 'SA1'...")

    # Write the filtered data to a new folder
    print(f"Writing SA1 data to {sa1_output_dir} ...")
    sa_ddf.to_parquet(sa1_output_dir, overwrite=True)
    print("SA1 filter complete!")

Loaded merged dataset from disk.
Filtering for Region == 'SA1'...
Writing SA1 data to /Volumes/T7/enriched_sa1 ...
SA1 filter complete!
